# MedQuAD Data Cleaning — MedlinePlus Health Topics

This notebook processes the **4_MPlus_Health_Topics_QA** subset of MedQuAD,
runs quality checks, normalizes text, and exports a clean CSV to `data/processed/`.

**Important:** Raw XML files are never modified. All cleaning happens in memory.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

import html
import re

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "MedQuAD" / "4_MPlus_Health_Topics_QA"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Output dir:   {PROCESSED_DIR}")

## 1. Discover XML files

Recursively find all `.xml` files under `data/raw/` instead of hardcoding a numeric range.

In [ ]:
xml_files = sorted(RAW_DIR.rglob("*.xml"))
print(f"Found {len(xml_files)} XML files")

## 2. Quality checks

Scan all QA pairs for empty fields, short text, and length distributions.

In [ ]:
empty_q = empty_a = short_q = short_a = total_pairs = parse_errors = 0
q_lengths, a_lengths = [], []

for filepath in xml_files:
    try:
        tree = ET.parse(filepath)
    except ET.ParseError:
        parse_errors += 1
        continue

    for qa in tree.getroot().iter("QAPair"):
        total_pairs += 1
        q = (qa.findtext("Question") or "").strip()
        a = (qa.findtext("Answer") or "").strip()

        empty_q += (q == "")
        empty_a += (a == "")
        short_q += (0 < len(q) < 5)
        short_a += (0 < len(a) < 10)

        if q:
            q_lengths.append(len(q))
        if a:
            a_lengths.append(len(a))

print(f"Total QA pairs: {total_pairs}")
print(f"Parse errors:   {parse_errors}")
print(f"Empty questions: {empty_q}, Empty answers: {empty_a}")
print(f"Short questions (<5 chars): {short_q}, Short answers (<10 chars): {short_a}")
print()
if q_lengths:
    print(f"Question length — mean: {np.mean(q_lengths):.1f}, min: {np.min(q_lengths)}, max: {np.max(q_lengths)}")
if a_lengths:
    print(f"Answer length   — mean: {np.mean(a_lengths):.1f}, p99: {np.percentile(a_lengths, 99):.0f}")

## 3. Parse and normalize

Extract all QA pairs into a DataFrame. Text normalization (HTML unescape,
whitespace collapsing) is applied in memory — raw XML files are **not** modified.

In [ ]:
def normalize_text(text: str) -> str:
    """HTML-unescape and collapse whitespace."""
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


rows = []

for filepath in xml_files:
    try:
        root = ET.parse(filepath).getroot()
    except ET.ParseError:
        continue

    doc_id = root.get("id", "")
    url = root.get("url", "")
    focus = (root.findtext("Focus") or "").strip()

    for qa in root.iter("QAPair"):
        q_el = qa.find("Question")
        q = normalize_text(q_el.text) if q_el is not None and q_el.text else ""
        a = normalize_text(qa.findtext("Answer") or "")
        qid = q_el.get("qid", "") if q_el is not None else ""

        if q and a:
            rows.append({
                "doc_id": doc_id,
                "focus": focus,
                "qid": qid,
                "question": q,
                "answer": a,
                "url": url,
                "source_file": filepath.name,
            })

df = pd.DataFrame(rows)
print(f"Extracted {len(df)} QA pairs")
df.head()

## 4. Deduplicate

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="question", keep="first").reset_index(drop=True)
print(f"Deduplicated: {before} -> {len(df)}")

## 5. Export

In [ ]:
csv_path = PROCESSED_DIR / "medquad_cleaned.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved {len(df)} rows to {csv_path}")

parquet_path = PROCESSED_DIR / "medquad_cleaned.parquet"
df.to_parquet(parquet_path, index=False)
print(f"Saved {len(df)} rows to {parquet_path}")